- D-Fire YOLO26x | Kaggle T4 x2
  - Input: Kaggle Dataset dfire-relabeled, tạo từ D-Fire.zip
  - Accelerator: GPU T4 x2; Internet on; không dùng P100
  - Protocol: COCO pretrained, split cố định trong ZIP, seed 20260707, 640 px, 20 epoch
  - Global batch: 8 ảnh/step, 4 ảnh/T4; accumulation 4; effective batch 32
  - Augmentation chung: horizontal flip 0.5; tắt augmentation khác
  - Resume: tự quét working + Kaggle Input; Input cần resume_protocol.json cùng run checkpoint
  - Output: /kaggle/working/runs/dfire_yolo26x/weights


In [ ]:
!find /kaggle/input/datasets -maxdepth 5 -type d | head -80


In [ ]:
from pathlib import Path

DATASET_SLUG = 'dfire-relabeled'
SEED = 20260707
EPOCHS = 20
RESOLUTION = 640

INPUT_BASE = Path('/kaggle/input/datasets')
WORK_ROOT = Path('/kaggle/working')
RUN_NAME = 'dfire_yolo26x'
GLOBAL_BATCH = 8
NOMINAL_BATCH = 32
RUN_DIR = WORK_ROOT / 'runs' / RUN_NAME


In [ ]:
import subprocess
import sys

subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.90'], check=True)

import torch
import ultralytics

devices = [torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())]
capabilities = [torch.cuda.get_device_capability(index) for index in range(torch.cuda.device_count())]
cuda_version = tuple(int(part) for part in torch.version.cuda.split('.')[:2])
print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'ultralytics': ultralytics.__version__, 'devices': devices, 'capabilities': capabilities})
assert len(devices) == 2 and all('T4' in name for name in devices), f'Cần Accelerator GPU T4 x2, hiện có {devices}'
assert all(value >= (7, 0) for value in capabilities), f'GPU không được torch {torch.__version__} hỗ trợ: {capabilities}'
assert cuda_version >= (12, 8), f'Cần Kaggle CUDA runtime >= 12.8, hiện có {torch.version.cuda}'


In [ ]:
from collections.abc import Mapping
import json, operator, os, shutil, tempfile, uuid
import yaml
PROTOCOL_NAME = 'resume_protocol.json'
PROTOCOL = {'schema': 1, 'framework': 'ultralytics-8.4.90', 'model': 'yolo26x', 'resolution': RESOLUTION, 'epochs': EPOCHS, 'seed': SEED, 'global_batch': GLOBAL_BATCH, 'nominal_batch': NOMINAL_BATCH}
weights_dir = RUN_DIR / 'weights'
working_run_present = RUN_DIR.exists()
dataset_candidates = list(INPUT_BASE.glob(f'*/{DATASET_SLUG}/{DATASET_SLUG}/D-Fire'))
if len(dataset_candidates) != 1: raise FileNotFoundError(f'Cần đúng 1 D-Fire root, tìm thấy {dataset_candidates}')
data_root = dataset_candidates[0]
for split in ('train', 'valid', 'test'):
    for child in ('images', 'labels'):
        if not (data_root / split / child).is_dir(): raise FileNotFoundError(data_root / split / child)
counts = {split: len(list((data_root / split / 'images').iterdir())) for split in ('train', 'valid', 'test')}
assert counts == {'train': 15500, 'valid': 1721, 'test': 4306}, counts
print({'data_root': str(data_root), 'counts': counts})
data_yaml = WORK_ROOT / 'dfire_yolo.yaml'
data_yaml.write_text(yaml.safe_dump({'path': str(data_root), 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'names': {0: 'smoke', 1: 'fire'}}, sort_keys=False), encoding='utf-8')
def atomic_copy(source, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=destination.parent, delete=False) as handle: temporary = Path(handle.name)
    try:
        shutil.copyfile(source, temporary)
        with temporary.open('rb+') as handle: handle.flush(); os.fsync(handle.fileno())
        os.replace(temporary, destination)
    finally: temporary.unlink(missing_ok=True)
def atomic_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', dir=path.parent, delete=False) as handle:
        temporary = Path(handle.name); json.dump(payload, handle, sort_keys=True); handle.flush(); os.fsync(handle.fileno())
    try: os.replace(temporary, path)
    finally: temporary.unlink(missing_ok=True)
def protocol_path(path, source):
    if source == 'working': return RUN_DIR / PROTOCOL_NAME
    for directory in (path.parent, *path.parents):
        if (directory / PROTOCOL_NAME).is_file(): return directory / PROTOCOL_NAME
        if directory == INPUT_BASE: break
def integer(value, name):
    if isinstance(value, bool): raise ValueError(f'{name} không phải int')
    return operator.index(value)
def validate(path, source):
    if path.stat().st_size < 1024 * 1024: raise ValueError('file quá nhỏ')
    state = torch.load(path, map_location='cpu', weights_only=False)
    if not isinstance(state, Mapping): raise ValueError('checkpoint không phải mapping')
    epoch, updates = integer(state.get('epoch'), 'epoch'), integer(state.get('updates'), 'updates')
    if epoch < 0 or updates < 0: raise ValueError('epoch hoặc updates âm')
    if any(state.get(key) is None for key in ('model', 'ema', 'optimizer', 'scaler')): raise ValueError('thiếu model/ema/optimizer/scaler')
    args = state.get('train_args')
    if not isinstance(args, Mapping): raise ValueError('thiếu train_args')
    if Path(str(args.get('model', ''))).stem.lower() != 'yolo26x': raise ValueError('model không khớp')
    for key, value in {'imgsz': RESOLUTION, 'epochs': EPOCHS, 'seed': SEED, 'batch': GLOBAL_BATCH}.items():
        if args.get(key) != value: raise ValueError(f'train_args.{key} không khớp')
    protocol = protocol_path(path, source)
    if protocol is None or json.loads(protocol.read_text(encoding='utf-8')) != PROTOCOL: raise ValueError('resume_protocol.json thiếu hoặc lệch')
    return {'path': path, 'source': source, 'epoch': epoch, 'step': updates}
def scan(paths, source):
    records=[]
    for path in sorted(set(paths)):
        try: records.append(validate(path, source))
        except Exception as error: print(f'reject {source}: {path}: {error}')
    return records
def checkpoint_name(path): return path.name in {'last.pt', 'last_resume.pt'} or path.name.startswith('epoch')
if working_run_present and not (RUN_DIR / PROTOCOL_NAME).is_file(): raise RuntimeError('Working run thiếu resume_protocol.json')
working = scan((p for p in weights_dir.glob('*.pt') if checkpoint_name(p)), 'working')
if working_run_present and not working: raise RuntimeError(f'Run directory có nhưng không checkpoint full-state hợp lệ: {RUN_DIR}')
input_records = scan((p for p in INPUT_BASE.rglob('*.pt') if checkpoint_name(p)), 'input')
selected = max(working + input_records, key=lambda r: (r['epoch'], r['step'], r['source'] == 'working'), default=None)
resume_path = None
if selected:
    if selected['source'] == 'input':
        name = 'input_epoch_{:03d}_updates_{:09d}.pt'.format(selected['epoch'], selected['step'])
        if working_run_present:
            resume_path = weights_dir / name
            if not resume_path.exists(): atomic_copy(selected['path'], resume_path)
        else:
            stage = RUN_DIR.parent / f'.{RUN_NAME}.stage-{uuid.uuid4().hex}'
            atomic_copy(selected['path'], stage / 'weights' / name); atomic_json(stage / PROTOCOL_NAME, PROTOCOL); os.replace(stage, RUN_DIR); resume_path = RUN_DIR / 'weights' / name
    else: resume_path = selected['path']
elif not working_run_present: atomic_json(RUN_DIR / PROTOCOL_NAME, PROTOCOL)
run_complete = bool(selected and selected['epoch'] >= EPOCHS - 1)
print({'resume_source': selected['source'] if selected else 'fresh', 'resume_path': str(resume_path) if resume_path else None, 'epoch': selected['epoch'] if selected else None, 'updates': selected['step'] if selected else None, 'complete': run_complete})


In [ ]:
from ultralytics import YOLO
if run_complete:
    print(f'Không train lại: checkpoint đã hoàn thành epoch {EPOCHS}.')
elif resume_path:
    model = YOLO(str(resume_path)); model.train(resume=str(resume_path), data=str(data_yaml), device=[0, 1])
else:
    model = YOLO('yolo26x.pt')
    model.train(data=str(data_yaml), project=str(RUN_DIR.parent), name=RUN_NAME, exist_ok=True, epochs=EPOCHS, imgsz=RESOLUTION, batch=GLOBAL_BATCH, nbs=NOMINAL_BATCH, device=[0, 1], workers=4, amp=True, seed=SEED, deterministic=True, patience=0, pretrained=True, cos_lr=True, lrf=0.01, warmup_epochs=0.0, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, degrees=0.0, translate=0.0, scale=0.0, shear=0.0, perspective=0.0, flipud=0.0, fliplr=0.5, bgr=0.0, mosaic=0.0, mixup=0.0, cutmix=0.0, close_mosaic=0, auto_augment=None, erasing=0.0, save_period=1)


In [ ]:
weights = sorted((RUN_DIR / 'weights').glob('*.pt'))
for path in weights:
    print(path, path.stat().st_size)
assert (RUN_DIR / 'weights' / 'best.pt').is_file()
assert (RUN_DIR / 'weights' / 'last.pt').is_file()
